# ASL Classifier — Letters + Numbers (grassknoted + ardamavi)

Trains a single Random Forest on:
- **Letters A–Y** (J and Z skipped — motion-based): `grassknoted/asl-alphabet` — 200×200 colour images
- **Numbers 0–9**: `ardamavi/sign-language-digits-dataset` — 64×64 grayscale images

Both datasets are processed through MediaPipe to extract 63-dim hand landmark vectors.

Output: `/kaggle/working/asl_classifier.pkl` — drop into `backend/`

---
## Add these datasets via `+ Add Data` before running:
1. Search **`grassknoted/asl-alphabet`**
2. Search **`ardamavi/sign-language-digits-dataset`**

In [ ]:
!pip install mediapipe --quiet

In [ ]:
import os, pickle, urllib.request
from pathlib import Path
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print('Imports OK')

In [ ]:
# Download MediaPipe hand landmarker model
HAND_MODEL_PATH = Path('/kaggle/working/hand_landmarker.task')
HAND_MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/'
    'hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
)
if not HAND_MODEL_PATH.exists():
    print('Downloading hand landmarker model...')
    urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL_PATH)
    print('Done.')
else:
    print('Already downloaded.')

base_options = mp_python.BaseOptions(model_asset_path=str(HAND_MODEL_PATH))
options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)
landmarker = mp_vision.HandLandmarker.create_from_options(options)

def extract_landmarks(img_path):
    """Run MediaPipe on an image file. Returns 63-dim wrist-normalised vector or None."""
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        return None
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)
    if not result.hand_landmarks:
        return None
    lm = result.hand_landmarks[0]
    wrist = lm[0]
    coords = []
    for point in lm:
        coords.extend([point.x - wrist.x, point.y - wrist.y, point.z - wrist.z])
    return np.array(coords, dtype=np.float32)

print('HandLandmarker ready.')

## Part 1 — Letters A–Y from grassknoted/asl-alphabet

In [ ]:
# Locate the asl-alphabet dataset
candidates = [
    '/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train',
    '/kaggle/input/asl-alphabet/asl_alphabet_train',
    '/kaggle/input/asl-alphabet',
]

LETTERS_ROOT = None
for c in candidates:
    p = Path(c)
    if p.exists():
        subdirs = [x.name for x in p.iterdir() if x.is_dir()]
        if 'A' in subdirs or 'a' in subdirs:
            LETTERS_ROOT = p
            break

if LETTERS_ROOT is None:
    print('Contents of /kaggle/input:')
    for p in sorted(Path('/kaggle/input').rglob('*')):
        if p.is_dir(): print(' ', p)
    raise FileNotFoundError('Add grassknoted/asl-alphabet via + Add Data')

print(f'Letters dataset: {LETTERS_ROOT}')
print('Subfolders:', sorted([x.name for x in LETTERS_ROOT.iterdir() if x.is_dir()]))

In [ ]:
SKIP_LETTERS = {'J', 'Z'}   # motion-based
MAX_PER_LETTER = 500

X_letters, y_letters = [], []
skipped_letters = 0

for letter_dir in sorted(LETTERS_ROOT.iterdir()):
    label = letter_dir.name.upper()
    if not letter_dir.is_dir() or label in SKIP_LETTERS or not label.isalpha() or len(label) != 1:
        continue

    images = (list(letter_dir.glob('*.jpg')) + list(letter_dir.glob('*.png')))[:MAX_PER_LETTER]
    count = 0
    for img_path in images:
        feat = extract_landmarks(img_path)
        if feat is not None:
            X_letters.append(feat)
            y_letters.append(label)
            count += 1
        else:
            skipped_letters += 1
    print(f'  {label}: {count}')

print(f'\nLetters total: {len(X_letters)}, skipped (no hand): {skipped_letters}')

## Part 2 — Numbers 0–9 from ardamavi/sign-language-digits-dataset

In [ ]:
# The ardamavi dataset has a Dataset/ folder with subfolders 0-9
candidates_num = [
    '/kaggle/input/sign-language-digits-dataset/Dataset',
    '/kaggle/input/sign-language-digits-dataset',
]

NUMBERS_ROOT = None
for c in candidates_num:
    p = Path(c)
    if p.exists():
        subdirs = [x.name for x in p.iterdir() if x.is_dir()]
        if any(s.isdigit() for s in subdirs):
            NUMBERS_ROOT = p
            break

if NUMBERS_ROOT is None:
    print('Contents of /kaggle/input:')
    for p in sorted(Path('/kaggle/input').rglob('*')):
        if p.is_dir(): print(' ', p)
    raise FileNotFoundError('Add ardamavi/sign-language-digits-dataset via + Add Data')

print(f'Numbers dataset: {NUMBERS_ROOT}')
print('Subfolders:', sorted([x.name for x in NUMBERS_ROOT.iterdir() if x.is_dir()]))

In [ ]:
MAX_PER_DIGIT = 500

X_numbers, y_numbers = [], []
skipped_numbers = 0

for digit_dir in sorted(NUMBERS_ROOT.iterdir()):
    label = digit_dir.name
    if not digit_dir.is_dir() or not label.isdigit():
        continue

    images = (list(digit_dir.glob('*.jpg')) +
              list(digit_dir.glob('*.png')) +
              list(digit_dir.glob('*.JPG')))[:MAX_PER_DIGIT]
    count = 0
    for img_path in images:
        feat = extract_landmarks(img_path)
        if feat is not None:
            X_numbers.append(feat)
            y_numbers.append(label)
            count += 1
        else:
            skipped_numbers += 1
    print(f'  {label}: {count}')

print(f'\nNumbers total: {len(X_numbers)}, skipped (no hand): {skipped_numbers}')

## Part 3 — Combine, Train, Evaluate, Save

In [ ]:
X_all = np.array(X_letters + X_numbers, dtype=np.float32)
y_all = np.array(y_letters + y_numbers)

print(f'Total samples : {len(X_all)}')
print(f'Classes ({len(set(y_all))}): {sorted(set(y_all))}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
print('Training Random Forest (300 trees, n_jobs=-1)...')
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train)
print('Training complete.')

In [ ]:
y_pred = clf.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.1%}')
print()
print(classification_report(y_test, y_pred))

In [ ]:
MODEL_OUT = Path('/kaggle/working/asl_classifier.pkl')
with open(MODEL_OUT, 'wb') as f:
    pickle.dump(clf, f)

print(f'Saved : {MODEL_OUT}')
print(f'Size  : {MODEL_OUT.stat().st_size / 1024 / 1024:.1f} MB')
print(f'Classes: {sorted(clf.classes_)}')
print()
print('NEXT STEPS:')
print('1. Download asl_classifier.pkl from the Output panel')
print('2. Replace backend/asl_classifier.pkl')
print('3. Restart the backend')